# Importing and preparing flatfox rental apartment data (buildings before 1991)

**This Jupyter notebook does the following:**

- Imports web-scraped flatfox rental apartment data (buildings built before 1991) from a CSV file
- Inspects the number of rows/columns and the data types
- Extracts the number of rooms, living area and rental price from raw strings using regular expressions (regex)
- Flags rents per week, furnished and temporary listings (they are kept, not deleted)
- Counts missing and duplicated values
- Topic 1 extensions: cleans construction/renovation years, computes rent per m², extracts postcode and locality, and assigns renovation groups (not renovated / general renovation / energy renovation) using keywords
- Saves the prepared data to `Data/flatfox_pre1991_prepared.csv`

The structure follows `apartments_data_preparation_winterthur.ipynb`.

## Libraries and settings

In [ ]:
# Libraries
import os
import re
import fnmatch
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

## Importing data

In [ ]:
# Show .csv - files in the folder 'Data'
flist = fnmatch.filter(os.listdir('./Data'), '*.csv')
for i in flist:
    print(i)

# Read the data to a pandas data frame
# (descriptions contain line breaks inside quotes; read_csv handles this)
df = pd.read_csv('./Data/flatfox_switzerland_rent_apartments_pre1991.csv', sep=',', encoding='utf-8')

# Show first records of data frame
df.head()

## Count number of rows and columns in the data frame

In [ ]:
# Dimension (rows, columns)
print('Dimension:', df.shape)

# Number of rows
print('Number of rows:', df.shape[0])

# Number of columns
print('Number of columns:', df.shape[1])

## Get data types (raw-format from web scraping)

In [ ]:
# Get data types (since pandas 3.0, text columns have the data type 'str';
# in older versions of pandas, they have the data type 'object')
df.dtypes

## Extract and save relevant information from raw data using regular expressions (regex)

### Extract number of rooms

The column `rooms_detail` contains values like `"3"` or `"3 ½"`. The text `"3 ½"` cannot be used in calculations, so the half room is converted to `.5` (e.g. `"3 ½"` → `3.5`).

In [ ]:
# Extract values from 'rooms_detail' strings
rooms = []
for i in df['rooms_detail'].fillna('').astype(str):
    d1 = re.findall(r'(\d+(?:[.,]\d+)?)\s*(½)?', i)
    try:
        d2 = float(d1[0][0].replace(',', '.')) + (0.5 if d1[0][1] else 0.0)
    except:
        d2 = None
    rooms.append(d2)

# Save as new variable in the pandas data frame
df['rooms'] = pd.Series(rooms, dtype="float64")

# Print first 5 values
print(df['rooms_detail'].head(5), '\n')
print(df['rooms'].head(5))

### Extract living area

In [ ]:
# Extract values from 'living_area' strings, e.g. '81 m²'
area = []
for i in df['living_area'].fillna('').astype(str):
    d1 = re.findall(r'(\d+)\s*m²', re.sub(r"['’]", '', i))  # remove thousands separators ' and ’
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    area.append(d2)

# Save as new variable in the pandas data frame
df['area'] = pd.Series(area, dtype="Int64")

# Print first 5 values
print(df['living_area'].head(5), '\n')
print(df['area'].head(5))

### Extract rental price

The column `gross_rent` contains values like `"CHF 1’200 per month"`. The thousands separator is the typographic apostrophe `’`, which is a different character from `'`. Both are removed, otherwise `"1’200"` would be read as `1`.

In [ ]:
# Extract values from 'gross_rent' strings
price = []
for i in df['gross_rent'].fillna('').astype(str):
    d1 = re.findall('[0-9]+', re.sub(r"[\s'’]", '', i))  # remove spaces and thousands separators ' and ’
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df['price'] = pd.Series(price, dtype="Int64")

# Print first 5 values
print(df['gross_rent'].head(5), '\n')
print(df['price'].head(5))

### Flag rent period, furnished and temporary listings

Some rents are given per week instead of per month. These rows are **kept** and only flagged with `rent_per_week`. Furnished and temporary listings are flagged as well, because they can bias the rent comparison.

In [ ]:
# Rent period ('month', 'week', or missing for 'By request')
df['rent_period'] = df['gross_rent'].str.extract(r'per\s+(\w+)', expand=False)
df['rent_per_week'] = df['rent_period'].eq('week')

# Furnished / temporary flags from 'price_card'
df['furnished'] = df['price_card'].str.contains('Furnished', na=False)
df['temporary'] = df['price_card'].str.contains('Temporary', na=False)

print(df['rent_period'].value_counts(dropna=False), '\n')
print('Furnished:', df['furnished'].sum(), '| Temporary:', df['temporary'].sum(), '\n')

# Show the listings with weekly rents
df.loc[df['rent_per_week'], ['address', 'gross_rent', 'price']]

### Get data types of all variables including the new ones

In [ ]:
df.dtypes

## Count and identify missing values (if any)

In [ ]:
# Count missing values
print(pd.isna(df).sum())

# Identify rows with missing values
df[df.isna().any(axis=1)].head()

## Count and identify duplicated values (if any)

- `detail_url-href` identifies a listing. Rows with the same URL were scraped twice and are removed.
- Same address, rooms and rent can also be **different** flats in the same building (e.g. two flats on different floors), so these are only shown, not removed.

In [ ]:
# Exact duplicates: same listing URL scraped more than once
print('Duplicated listing URLs:', df.duplicated(subset=['detail_url-href']).sum())

# Possible duplicates: same address, rooms and rent
dup_cols = ['address', 'rooms_detail', 'gross_rent']
print('Same address, rooms and rent:', df.duplicated(subset=dup_cols).sum())

# Identify rows with possible duplicates (all occurrences)
df[df.duplicated(subset=dup_cols, keep=False)].sort_values('address')[dup_cols + ['detail_url-href']]

In [ ]:
# Remove exact duplicates (same listing URL)
df = df.drop_duplicates(subset=['detail_url-href']).reset_index(drop=True)
print('Dimension after removing duplicated URLs:', df.shape)

## Topic 1: additional preparation

The following steps are not part of the Week 8 example. They are needed for the research question of Topic 1 (green premium of energy-renovated older apartments in Switzerland).

### Clean construction year and renovation year

- `year_built`: values of `0` or below 1000 (e.g. `189`) are set to missing. Only years up to 1990 are kept, as in the scrape filter.
- `year_renovated`: values of `0`, years in the future (after 2026) and years not after the construction year are set to missing.

In [ ]:
# Convert to numbers (non-numeric values become missing)
df['year_built'] = pd.to_numeric(df['year_built'], errors='coerce')
df['year_renovated'] = pd.to_numeric(df['year_renovated'], errors='coerce')

# Clean construction year
df['year_built_clean'] = df['year_built'].where(df['year_built'].between(1000, 1990))

# Clean renovation year
valid_reno = (df['year_renovated'].between(1900, 2026)
              & (df['year_renovated'] > df['year_built_clean'].fillna(0)))
df['year_renovated_clean'] = df['year_renovated'].where(valid_reno)

# Show the values that were set to missing
print('year_built set to missing:')
print(df.loc[df['year_built'].notna() & df['year_built_clean'].isna(), ['address', 'year_built']], '\n')
print('year_renovated set to missing:')
print(df.loc[df['year_renovated'].notna() & df['year_renovated_clean'].isna(),
             ['address', 'year_built', 'year_renovated']])

### Plausibility check of living area and rent per m²

- Living areas below 10 m² or above 500 m² are set to missing (e.g. `1097 m²` is very likely a typo).
- `rent_per_m2` = gross rent / living area. It is only computed for **monthly** rents; for weekly rents it stays empty (the rows are kept and flagged).
- Note: the data only contains **gross** rent (including Nebenkosten), not net rent.

In [ ]:
# Plausible living area
df['area_clean'] = df['area'].where(df['area'].between(10, 500))
print('Area set to missing:')
print(df.loc[df['area'].notna() & df['area_clean'].isna(), ['address', 'living_area']], '\n')

# Gross rent per m² (monthly rents only)
monthly = df['rent_period'].eq('month')
df['rent_per_m2'] = (df['price'] / df['area_clean']).where(monthly).round(2)

print(df['rent_per_m2'].describe())

### Extract postcode and locality

The postcode and locality are extracted from `address` (e.g. `'8400 Winterthur'`). They are needed later to merge municipality-level data (e.g. SFSO "Regionalporträts", which covers all Swiss municipalities) as control variables for location.

In [ ]:
# Postcode and locality from 'address', e.g. '8400 Winterthur'
df['zip'] = pd.to_numeric(df['address'].str.extract(r'^(\d{4})', expand=False), errors='coerce').astype('Int64')
df['locality'] = df['address'].str.extract(r'^\d{4}\s+(.*)$', expand=False).str.strip()

print('Listings without postcode:', df['zip'].isna().sum())
print('Number of different localities:', df['locality'].nunique(), '\n')

# Most frequent localities
print(df['locality'].value_counts().head(10))

### Renovation groups (keyword search in the description)

Each listing is assigned to one group:

1. **energy renovation**: the description contains an energy keyword (e.g. Wärmepumpe, Minergie, new windows, insulation)
2. **general renovation**: a general renovation keyword (e.g. renoviert, neue Küche) **or** a valid renovation year
3. **not renovated**: none of the above

The descriptions are in German, French and Italian, so keywords in all three languages are used. Keywords are matched as lower-case substrings. This is a simple heuristic: listings may be assigned to the wrong group (see limitations of Topic 1). Review a sample of each group and adjust the keyword lists if needed.

In [ ]:
energy_keywords = [
    # German
    'energetisch', 'minergie', 'wärmepumpe', 'fassade', 'gebäudehülle', 'dämmung', 'gedämmt',
    'neue fenster', 'fenster ersetzt', 'fenstersanierung', '3-fach', '3fach', 'dreifach',
    'photovoltaik', 'solaranlage', 'pellet', 'neue heizung', 'heizung ersetzt',
    # French
    'pompe à chaleur', 'isolation thermique', 'façade', 'triple vitrage', 'photovoltaïque',
    'panneaux solaires', 'haute performance énergétique',
    # Italian
    'pompa di calore', 'facciate isolate', 'cappotto', 'tripli vetri', 'triplo vetro', 'fotovoltaico',
]

general_keywords = [
    # German
    'renoviert', 'renovation', 'saniert', 'sanierung', 'erneuert', 'modernisiert', 'umgebaut',
    'neue küche', 'neues bad', 'neue bad',
    # French
    'rénové', 'rénovation', 'refait', 'cuisine neuve',
    # Italian
    'ristrutturat', 'rinnovat', 'risanament',
]

desc = df['description'].fillna('').str.lower()
df['kw_energy'] = desc.apply(lambda t: any(k in t for k in energy_keywords))
df['kw_general'] = desc.apply(lambda t: any(k in t for k in general_keywords))

df['renovation_group'] = np.select(
    [df['kw_energy'],
     df['kw_general'] | df['year_renovated_clean'].notna()],
    ['energy renovation', 'general renovation'],
    default='not renovated')

print(df['renovation_group'].value_counts(), '\n')

# Mean gross rent per m² by renovation group (first descriptive comparison, no controls)
df.groupby('renovation_group')['rent_per_m2'].agg(['count', 'mean', 'median']).round(2)

In [ ]:
# Review: listings in the energy renovation group and the keywords found
def found_keywords(text, keywords):
    return ', '.join(k for k in keywords if k in text)

review = df[df['renovation_group'] == 'energy renovation'][['address', 'year_renovated_clean']].copy()
review['energy_keywords_found'] = desc[review.index].apply(lambda t: found_keywords(t, energy_keywords))
review

### Save data to file

In [ ]:
df.to_csv('./Data/flatfox_pre1991_prepared.csv',
          sep=",",
          encoding='utf-8',
          index=False)

print('Saved:', df.shape[0], 'listings')

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')